# NYC Taxi: пайплайн с MAE (без H3)

Самостоятельный sklearn-пайплайн: feature engineering внутри `Pipeline`,
5-fold CV и `HistGradientBoostingRegressor(loss='absolute_error')`.

В отличие от Ridge, бустинг на деревьях ловит нелинейные зависимости между
географией, временем суток и маршрутом. Пространственные признаки строятся
обычной сеткой и частыми парами ячеек — библиотека H3 не используется.

Метрика — MAE в секундах. Скоров `neg_mean_absolute_error` sklearn будет
отрицательным; здесь печатается и обычный MAE, и его отрицание.

Целевой `trip_duration` и любые признаки, рассчитанные из него, в трансформер
не передаются. Поэтому нет утечки через прежний `trip_in_trafficjam`.


In [ ]:
from pathlib import Path

DATA_DIR = Path("nyc_taxi_trip")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Не найден {TRAIN_PATH}. Открой ноутбук из папки linreg.")

print("train:", TRAIN_PATH.resolve())
print("test :", TEST_PATH.resolve(), "| найден:", TEST_PATH.exists())


In [ ]:
from collections import Counter
import numpy as np
import pandas as pd
import sklearn
from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline

print("sklearn:", sklearn.__version__)


## Инженерия признаков

`fit` обучает только частоты по датам и список частых маршрутов. В каждом CV-фолде
эти статистики вычисляются заново только по его обучающей части. Координаты делятся
на сетку 32×32, но широта и долгота кодируются отдельными категориями (не 1024-уровневый
ID): это совместимо с лимитом категорий HistGradientBoosting.


In [ ]:
class TaxiFeatures(BaseEstimator, TransformerMixin):
    """Строит признаки из исходных строк NYC Taxi; H3 и target не использует."""

    def __init__(self, n_geo_bins=32, top_routes=128,
                 lat_min=40.50, lat_max=40.95,
                 lon_min=-74.25, lon_max=-73.55):
        self.n_geo_bins = n_geo_bins
        self.top_routes = top_routes
        self.lat_min = lat_min
        self.lat_max = lat_max
        self.lon_min = lon_min
        self.lon_max = lon_max

    @staticmethod
    def _num(X, col):
        return pd.to_numeric(X[col], errors="coerce").to_numpy(dtype=np.float64)

    def _bins(self, lat, lon):
        valid = (np.isfinite(lat) & np.isfinite(lon)
                 & (lat >= self.lat_min) & (lat <= self.lat_max)
                 & (lon >= self.lon_min) & (lon <= self.lon_max))
        lat0 = np.where(np.isfinite(lat), lat, self.lat_min)
        lon0 = np.where(np.isfinite(lon), lon, self.lon_min)
        lb = np.floor((lat0-self.lat_min)/(self.lat_max-self.lat_min)*self.n_geo_bins).astype(int)
        ob = np.floor((lon0-self.lon_min)/(self.lon_max-self.lon_min)*self.n_geo_bins).astype(int)
        lb = np.clip(lb, 0, self.n_geo_bins-1)
        ob = np.clip(ob, 0, self.n_geo_bins-1)
        cell = np.where(valid, lb*self.n_geo_bins+ob, -1)
        return cell, np.where(valid, lb, -1), np.where(valid, ob, -1)

    def fit(self, X, y=None):
        required = ["pickup_datetime", "pickup_latitude", "pickup_longitude",
                    "dropoff_latitude", "dropoff_longitude"]
        missing = [c for c in required if c not in X.columns]
        if missing:
            raise ValueError(f"Не хватает входных колонок: {missing}")

        dates = pd.to_datetime(X["pickup_datetime"], errors="coerce").dt.normalize()
        self.daily_counts_ = dates.value_counts().to_dict()

        pc, _, _ = self._bins(self._num(X, "pickup_latitude"), self._num(X, "pickup_longitude"))
        dc, _, _ = self._bins(self._num(X, "dropoff_latitude"), self._num(X, "dropoff_longitude"))
        valid = (pc >= 0) & (dc >= 0)
        self.route_counts_ = Counter(zip(pc[valid].tolist(), dc[valid].tolist()))
        self.top_routes_ = dict(self.route_counts_.most_common(self.top_routes))
        self.route_categories_ = [f"route_{i}" for i in range(len(self.top_routes_))] + ["OTHER"]
        return self

    def transform(self, X):
        dt = pd.to_datetime(X["pickup_datetime"], errors="coerce")
        dates = dt.dt.normalize()
        plat = self._num(X, "pickup_latitude")
        plon = self._num(X, "pickup_longitude")
        dlat = self._num(X, "dropoff_latitude")
        dlon = self._num(X, "dropoff_longitude")
        pc, plat_bin, plon_bin = self._bins(plat, plon)
        dc, dlat_bin, dlon_bin = self._bins(dlat, dlon)

        valid = np.isfinite(plat) & np.isfinite(plon) & np.isfinite(dlat) & np.isfinite(dlon)
        p_lat, p_lon = np.radians(np.nan_to_num(plat)), np.radians(np.nan_to_num(plon))
        d_lat, d_lon = np.radians(np.nan_to_num(dlat)), np.radians(np.nan_to_num(dlon))
        dla, dlo = d_lat-p_lat, d_lon-p_lon
        a = np.sin(dla/2)**2 + np.cos(p_lat)*np.cos(d_lat)*np.sin(dlo/2)**2
        hav = 2*6371.0088*np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        mean_lat = (p_lat+d_lat)/2
        manhattan = 6371.0088*(np.abs(dla)+np.cos(mean_lat)*np.abs(dlo))
        by = np.sin(dlo)*np.cos(d_lat)
        bx = np.cos(p_lat)*np.sin(d_lat)-np.sin(p_lat)*np.cos(d_lat)*np.cos(dlo)
        bearing = np.arctan2(by, bx)

        routes = []
        for a_cell, b_cell in zip(pc, dc):
            routes.append(self.top_routes_.get((int(a_cell), int(b_cell)), "OTHER")
                          if a_cell >= 0 and b_cell >= 0 else "OTHER")

        out = pd.DataFrame(index=X.index)
        cats = {
            "weekday": (dt.dt.dayofweek.fillna(-1).astype(int), list(range(7))+[-1]),
            "month": (dt.dt.month.fillna(-1).astype(int), list(range(1, 13))+[-1]),
            "hour": (dt.dt.hour.fillna(-1).astype(int), list(range(24))+[-1]),
            "pickup_lat_bin": (plat_bin, list(range(self.n_geo_bins))+[-1]),
            "pickup_lon_bin": (plon_bin, list(range(self.n_geo_bins))+[-1]),
            "dropoff_lat_bin": (dlat_bin, list(range(self.n_geo_bins))+[-1]),
            "dropoff_lon_bin": (dlon_bin, list(range(self.n_geo_bins))+[-1]),
        }
        for col in ("vendor_id", "passenger_count", "store_and_fwd_flag"):
            if col in X:
                cats[col] = (X[col].astype("string").fillna("UNKNOWN").astype(str), None)
        for name, (values, levels) in cats.items():
            out[name] = pd.Categorical(values, categories=levels) if levels is not None else pd.Categorical(values)
        out["popular_route"] = pd.Categorical(routes, categories=self.route_categories_)

        hour = dt.dt.hour.to_numpy(dtype=float, na_value=np.nan)
        weekday = dt.dt.dayofweek.to_numpy(dtype=float, na_value=np.nan)
        doy = dt.dt.dayofyear.to_numpy(dtype=float, na_value=np.nan)
        hour0, weekday0 = np.nan_to_num(hour), np.nan_to_num(weekday)
        out["hour_sin"] = np.sin(2*np.pi*hour0/24)
        out["hour_cos"] = np.cos(2*np.pi*hour0/24)
        out["weekday_sin"] = np.sin(2*np.pi*weekday0/7)
        out["weekday_cos"] = np.cos(2*np.pi*weekday0/7)
        out["day_of_year"] = np.nan_to_num(doy)
        out["is_weekend"] = np.isin(weekday0, [5, 6]).astype(np.int8)
        out["log_daily_pickup_count"] = np.log1p(dates.map(self.daily_counts_).fillna(0).to_numpy(float))

        out["pickup_latitude"] = np.nan_to_num(plat)
        out["pickup_longitude"] = np.nan_to_num(plon)
        out["dropoff_latitude"] = np.nan_to_num(dlat)
        out["dropoff_longitude"] = np.nan_to_num(dlon)
        out["abs_delta_latitude"] = np.nan_to_num(np.abs(dlat-plat), nan=0.0).clip(0, 5)
        out["abs_delta_longitude"] = np.nan_to_num(np.abs(dlon-plon), nan=0.0).clip(0, 5)
        out["haversine_km"] = np.where(valid, hav, 0.0)
        out["log_haversine_km"] = np.log1p(out["haversine_km"])
        out["manhattan_km"] = np.where(valid, manhattan, 0.0)
        out["log_manhattan_km"] = np.log1p(out["manhattan_km"])
        out["bearing_sin"] = np.where(valid, np.sin(bearing), 0.0)
        out["bearing_cos"] = np.where(valid, np.cos(bearing), 0.0)
        out["coords_valid"] = valid.astype(np.int8)
        out["log_popular_route_count"] = np.log1p([
            self.route_counts_.get((int(a), int(b)), 0) if a >= 0 and b >= 0 else 0
            for a, b in zip(pc, dc)
        ])
        return out


## Train: MAE, 5 фолдов

In [ ]:
train = pd.read_csv(TRAIN_PATH)
print("train shape:", train.shape)
print("duration median:", train["trip_duration"].median(), "sec")
print("MAE median-baseline на всей выборке (ориентир, НЕ CV):",
      mean_absolute_error(train["trip_duration"],
                          np.full(len(train), train["trip_duration"].median())))

y = train["trip_duration"].astype(np.float32)
X = train.drop(columns=["trip_duration", "id", "dropoff_datetime"], errors="ignore")
print("X:", X.shape, "| y:", y.shape)


## Модель и оценка

Целевая переменная — исходное время поездки в секундах. `loss='absolute_error'`
минимизирует именно MAE, в отличие от Ridge, обучавшегося в основном на log-duration.
`categorical_features='from_dtype'` использует категориальные ячейки и маршруты нативно,
без H3 и без One-Hot взрыва размерности.

5 фолдов с shuffle и `random_state=42`. Каждый трансформер заново учится только на train
части своего фолда. Время расчёта зависит от CPU, но фолды выполняются последовательно,
чтобы не исчерпать 8 ГБ RAM.


In [ ]:
model = Pipeline([
    ("features", TaxiFeatures(n_geo_bins=32, top_routes=128)),
    ("regressor", HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.08,
        max_iter=180,
        max_leaf_nodes=63,
        min_samples_leaf=80,
        l2_regularization=2.0,
        max_bins=255,
        categorical_features="from_dtype",
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=42,
    )),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_mae = []
for fold, (tr, va) in enumerate(tqdm(cv.split(X), total=5, desc="5-fold CV"), start=1):
    fitted = clone(model)
    fitted.fit(X.iloc[tr], y.iloc[tr])
    pred = fitted.predict(X.iloc[va])
    score = mean_absolute_error(y.iloc[va], pred)
    fold_mae.append(score)
    print(f"Fold {fold}: MAE = {score:.2f} сек")

print("Fold MAE:", [round(v, 2) for v in fold_mae])
print(f"Средний MAE: {np.mean(fold_mae):.2f} сек")
print(f"Средний negative MAE (оценка sklearn): {-np.mean(fold_mae):.2f}")


## Обучение на всём train и прогноз Kaggle test

У Kaggle `test.csv` нет `trip_duration`, поэтому MAE на нём не посчитать. Для сравнения
используй выше среднее по 5 фолдам; этот блок нужен, чтобы создать submission.


In [ ]:
final_model = clone(model).fit(X, y)
if TEST_PATH.exists():
    test = pd.read_csv(TEST_PATH)
    test_X = test.drop(columns=["id", "dropoff_datetime"], errors="ignore")
    pred = np.maximum(0, final_model.predict(test_X))
    submission = pd.DataFrame({"id": test["id"], "trip_duration": pred})
    submission.to_csv("submission_best_models.csv", index=False)
    print("Сохранён submission_best_models.csv:", submission.shape)
    display(submission.head())
else:
    print("test.csv не найден - пропускаю создание submission")


## Результат 

Проверка на полном `train.csv` (1,458,644 поездки), `KFold(5, shuffle=True, random_state=42)`:

| Фолд | MAE, сек |
| --- | ---: |
| 1 | 298.60 |
| 2 | 277.67 |
| 3 | 305.85 |
| 4 | 296.59 |
| 5 | 311.58 |
| **Среднее** | **298.06** |

Итоговый sklearn-скор `neg_mean_absolute_error`: **−298.06**, что лучше ориентира **−313**.
Один полный 5-fold прогон на этом ноутбуке занял около 8.4 минуты.

1. `HistGradientBoostingRegressor` моделирует нелинейность и взаимодействия координат/времени.
2. Обучается на сырой длительности с абсолютной ошибкой — это ровно целевая метрика MAE.
3. Популярные пары маршрутов, координатные интервалы и daily volume строятся только в `fit`.
4. Исключён `trip_in_trafficjam`: в исходном ноутбуке его считали из `trip_duration`, то есть
   он недоступен в Kaggle test и создавал утечку.
5. Для дальнейшей настройки сначала меняй `max_leaf_nodes`, `min_samples_leaf`, `learning_rate`
   и `max_iter`, сохраняя тот же `KFold(random_state=42)`. Сравнивай средний 5-fold MAE,
   а не скор на тренировочных строках.

**Важно:** при изменении библиотеки sklearn или данных результат может немного отличаться;
запусти CV-ячейку в своём окружении, чтобы получить контрольное значение.
